<div style="background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 20px; border-radius: 10px; color: white; margin-bottom: 20px;">
  <h1 style="margin:0; font-family: monospace;">🔋 Separator Sample Batch Registration</h2>
  <p style="margin:4px 0 0; opacity:0.6; font-size:0.85rem;">Voila Dashboard for a batch registration of Separator Sample entries in NOMAD</p>
</div>

In [ ]:
SPINNER_HTML = '''
<div style="display:flex;align-items:center;gap:12px;background:#fff3cd;padding:12px 16px;border-radius:6px">
  <div style="width:22px;height:22px;border:3px solid #ffc107;border-top-color:transparent;
              border-radius:50%;animation:spin 0.8s linear infinite"></div>
  <span>{msg}</span>
</div>
<style>@keyframes spin {{ to {{ transform: rotate(360deg); }} }}</style>
'''

import io
import os
import json
import time
import zipfile
import sys
from datetime import datetime
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd

try:
    import requests
    import urllib3
    import ipywidgets as widgets
    from ipyaggrid import Grid
    from IPython.display import display, HTML, clear_output
    from urllib3.exceptions import InsecureRequestWarning
    urllib3.disable_warnings(InsecureRequestWarning)
except ImportError as e:
    print(f"⚠️  Import Error: {e}")
    raise

# Import utils
from pathlib import Path
notebook_dir = Path('.').resolve()
sys.path.insert(0, str(notebook_dir.parent))
from utils import (
    NOMADAPIClient,
    clean_text,
    normalize_value,
    normalize_sample_id,
    coerce_value,
    make_row_by_key,
    wait_for_sample_ids,
)

# Add custom CSS
display(HTML("""
    <style>
        /* General styling improvements */
        .jupyter-widgets {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        
        /* Button styling */
        .widget-button {
            background: linear-gradient(135deg, #274b8e 0%, #3b6db0 100%) !important;
            border: none !important;
            border-radius: 8px !important;
            color: white !important;
            font-weight: 600 !important;
            transition: all 0.3s ease !important;
            box-shadow: 0 4px 15px rgba(102, 126, 234, 0.3) !important;
        }
        
        .widget-button:hover {
            transform: translateY(-2px) !important;
            box-shadow: 0 8px 25px rgba(102, 126, 234, 0.4) !important;
        }
        
        /* Dropdown styling */
        .widget-dropdown select {
            border: 2px solid #e1e8ed !important;
            border-radius: 8px !important;
            padding: 8px 12px !important;
            font-size: 14px !important;
        }
        
        /* Output area styling */
        .jupyter-widgets-output-area {
            background: #f8f9fa;
            border-radius: 10px;
            padding: 15px;
            margin: 10px 0;
            border-left: 4px solid #28a745;
        }
        
        /* AG Grid custom styling */
        

    </style>
"""))

# API Configuration
URL_BASE = 'https://nomad08.csn29.bessy.de'
URL = f'{URL_BASE}/nomad-oasis/api/v1'
TOKEN = os.environ.get('NOMAD_CLIENT_ACCESS_TOKEN', '')
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'Accept': 'application/json',
}

# Initialize API client
api_client = NOMADAPIClient(URL_BASE, HEADERS)

PROCESS_SETTLE_SECONDS = 2
PROCESS_RETRY_COUNT = 2

PLUGIN_PREFIX = 'nomad_battery_space.schema_packages.hzb_bs_package'

In [ ]:
# Base schema definition for SeparatorSample
BASE_SCHEMA = {
    'entry_type': 'Separator Sample',
    'm_def': f'{PLUGIN_PREFIX}.SeparatorSample',
    'required_keys': ['name'],
}

# Base columns (always visible)
BASE_COLUMNS = [
    ('name', 'Sample Name *', 'str'),
    ('separator_stock', 'Separator Stock (Reference)', 'str'),
    ('thickness_um', 'Thickness [µm]', 'float'),
    ('mass_g', 'Mass [g]', 'float'),
    ('description', 'Description', 'str'),
]

# Shape-specific columns
SHAPE_COLUMNS = {
    'Circle': [
        ('diameter_mm', 'Diameter [mm]', 'float'),
        ('area_cm2', 'Area [cm²]', 'float'),
    ],
    'Rectangle': [
        ('length_mm', 'Length [mm]', 'float'),
        ('width_mm', 'Width [mm]', 'float'),
        ('area_cm2', 'Area [cm²]', 'float'),
    ],
    'Other': [
        ('shape_description', 'Shape Description', 'str'),
    ],
}

# Product info columns (optional)
PRODUCT_INFO_COLUMNS = [
    ('supplier', 'Supplier', 'str'),
    ('product_number', 'Product Number', 'str'),
]

def get_columns_for_shape_and_product_info(shape, include_product_info):
    """Build column list based on shape and product info preference."""
    columns = BASE_COLUMNS.copy()
    
    if shape in SHAPE_COLUMNS:
        columns.extend(SHAPE_COLUMNS[shape])
    
    if include_product_info:
        columns.extend(PRODUCT_INFO_COLUMNS)
    
    return columns

# Initial schema (will be updated dynamically)
SCHEMAS = {
    'Separator Sample': {**BASE_SCHEMA, 'columns': BASE_COLUMNS},
}

In [ ]:
# API wrapper functions 
def api_get(path, params=None):
    return requests.get(f'{api_client.url}{path}', headers=HEADERS, params=params, verify=False)

def api_post(path, json_body=None):
    return requests.post(f'{api_client.url}{path}', headers=HEADERS, json=json_body, verify=False)

def iter_archive_query(query, required=None, owner='visible', page_size=200):
    required = required or {'data': '*', 'metadata': '*'}
    page_after_value = None
    results = []

    while True:
        body = {
            'required': required,
            'owner': owner,
            'query': query,
            'pagination': {'page_size': page_size},
        }
        if page_after_value:
            body['pagination']['page_after_value'] = page_after_value

        response = api_post('/entries/archive/query', json_body=body)
        response.raise_for_status()
        payload = response.json()
        results.extend(payload.get('data', []))

        pagination = payload.get('pagination', {})
        page_after_value = pagination.get('next_page_after_value')
        if not page_after_value:
            break

    return results

def get_uploads():
    response = api_get('/uploads', params={'page_size': 200})
    response.raise_for_status()
    return response.json().get('data', [])

def get_upload_id(name):
    for upload in uploads:
        if upload.get('upload_name') == name:
            return upload.get('upload_id')
    return None

def matches_schema(entry, schema):
    archive = entry.get('archive', {})
    data = archive.get('data', {})
    metadata = archive.get('metadata', {})
    return (
        data.get('m_def') == schema['m_def']
        or metadata.get('entry_type') == schema['entry_type']
    )

def get_existing_samples(upload_id, schema):
    if not upload_id:
        return []

    all_entries = iter_archive_query({'upload_id': upload_id})
    filtered = [entry for entry in all_entries if matches_schema(entry, schema)]
    filtered.sort(key=lambda item: item.get('archive', {}).get('data', {}).get('name', ''))
    return filtered

def get_raw_path_metadata(upload_id, raw_path):
    encoded_path = quote(raw_path.strip('/'), safe='/')
    response = api_get(f'/uploads/{upload_id}/rawdir/{encoded_path}')
    if response.status_code == 404:
        return None
    response.raise_for_status()
    payload = response.json()
    data = payload.get('data', payload)
    return data

def build_archive_bundle(prepared_rows):
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, mode='w', compression=zipfile.ZIP_DEFLATED) as archive_zip:
        for item in prepared_rows:
            archive_zip.writestr(item['raw_path'], json.dumps(item['archive'], indent=2))
    buffer.seek(0)
    return buffer.getvalue()

def write_archive_bundle_via_api(upload_id, prepared_rows):
    conflicting_paths = []
    for item in prepared_rows:
        existing_target = get_raw_path_metadata(upload_id, item['raw_path'])
        if existing_target and existing_target.get('directory_metadata'):
            conflicting_paths.append(item['raw_path'])

    if conflicting_paths:
        return 409, (
            'These raw paths already exist as directories from an earlier malformed upload: '
            + ', '.join(conflicting_paths)
            + '. Delete those sample paths in NOMAD or use a fresh upload, then retry.'
        )

    bundle = build_archive_bundle(prepared_rows)
    response = requests.put(
        f'{api_client.url}/uploads/{upload_id}/raw/',
        headers=HEADERS,
        params={
            'overwrite_if_exists': 'true',
            'auto_decompress': 'true',
        },
        files={
            'file': (
                'separator_batch_upload.zip',
                bundle,
                'application/zip',
            )
        },
        verify=False,
    )
    detail = response.text
    try:
        detail = response.json()
    except ValueError:
        pass
    return response.status_code, detail

def process_upload(upload_id, timeout=180):
    response = api_post(f'/uploads/{upload_id}/action/process')
    last_payload = None

    if response.status_code not in (200, 202):
        detail = response.text
        try:
            detail = response.json()
        except ValueError:
            pass

        state_response = api_get(f'/uploads/{upload_id}')
        state_response.raise_for_status()
        last_payload = state_response.json().get('data', {})

        if not (response.status_code == 400 and last_payload.get('process_running')):
            raise requests.HTTPError(
                f'Processing failed: {response.status_code} {detail}',
                response=response,
            )

    deadline = time.time() + timeout
    while time.time() < deadline:
        time.sleep(1)
        state_response = api_get(f'/uploads/{upload_id}')
        state_response.raise_for_status()
        last_payload = state_response.json().get('data', {})
        if not last_payload.get('process_running'):
            return last_payload

    raise TimeoutError(f'Processing did not finish within {timeout} seconds.')

def get_upload_processing_details(upload_id):
    details = {}
    page_after_value = None
    while True:
        params = {'page_size': 200}
        if page_after_value:
            params['page_after_value'] = page_after_value

        response = api_get(f'/uploads/{upload_id}/entries', params=params)
        response.raise_for_status()
        payload = response.json()
        
        for entry in payload.get('data', []):
            mainfile = entry.get('mainfile', '')
            if mainfile:
                details[mainfile] = entry
                details[Path(mainfile).name] = entry
        
        pagination = payload.get('pagination', {})
        page_after_value = pagination.get('next_page_after_value')
        if not page_after_value:
            break

    return details


In [ ]:
def validate_row(schema_name, schema, row_by_key):
    """Validate a row against schema requirements."""
    missing = [key for key in schema['required_keys'] if not row_by_key.get(key)]
    if missing:
        missing_labels = []
        for key in missing:
            for column_key, column_label, _dtype in schema['columns']:
                if column_key == key:
                    missing_labels.append(column_label)
                    break
        return f"Missing required fields: {', '.join(missing_labels)}"
    return None

def extract_grid_frame():
    """Extract the current grid data as a DataFrame."""
    if grid is None:
        return pd.DataFrame(columns=col_labels)

    grid_data_out = getattr(grid, 'grid_data_out', {}) or {}
    candidate = grid_data_out.get('grid') if isinstance(grid_data_out, dict) else None

    if isinstance(candidate, pd.DataFrame):
        return candidate.copy().reset_index(drop=True)
    if isinstance(candidate, list):
        return pd.DataFrame(candidate).reset_index(drop=True)
    if isinstance(getattr(grid, 'grid_data', None), pd.DataFrame):
        return grid.grid_data.copy().reset_index(drop=True)
    return pd.DataFrame(columns=col_labels)

def prepare_rows_from_frame(frame, schema_name, schema, existing_ids):
    """Prepare rows from grid data for upload with dynamic shape handling."""
    prepared_rows = []
    row_errors = []
    
    selected_shape = shape_dd.value
    include_product_info = product_info_cb.value

    for row_number, (_row_index, row) in enumerate(frame.iterrows(), start=1):
        row_by_key = make_row_by_key(row, col_keys, col_labels)
        if not any(value is not None for value in row_by_key.values()):
            continue

        validation_error = validate_row(schema_name, schema, row_by_key)
        if validation_error:
            sample_name = row_by_key.get('name') or f'row {row_number}'
            row_errors.append(f'{sample_name}: {validation_error}')
            continue

        sample_name = normalize_sample_id(row_by_key.get('name'))

        # Build dimensions_and_weights subsection
        dimensions_and_weights = {}
        if row_by_key.get('thickness_um'):
            dimensions_and_weights['thickness'] = float(row_by_key['thickness_um'])
        if row_by_key.get('mass_g'):
            dimensions_and_weights['mass'] = float(row_by_key['mass_g'])
        
        # Build nested shape subsection based on selected shape
        shape = None
        if selected_shape == 'Circle':
            if row_by_key.get('diameter_mm'):
                shape = {
                    'm_def': f'{PLUGIN_PREFIX}.CircleGeometry',
                    'diameter': float(row_by_key['diameter_mm'])
                }
        elif selected_shape == 'Rectangle':
            if row_by_key.get('length_mm') or row_by_key.get('width_mm'):
                shape = {'m_def': f'{PLUGIN_PREFIX}.RectangleGeometry'}
                if row_by_key.get('length_mm'):
                    shape['length'] = float(row_by_key['length_mm'])
                if row_by_key.get('width_mm'):
                    shape['width'] = float(row_by_key['width_mm'])
        elif selected_shape == 'Other':
            if row_by_key.get('shape_description'):
                shape = {
                    'm_def': f'{PLUGIN_PREFIX}.OtherGeometry',
                    'description': row_by_key['shape_description']
                }
        
        if shape:
            dimensions_and_weights['shape'] = shape

        # Build product_info subsection (if enabled)
        product_info = {}
        if include_product_info:
            if row_by_key.get('supplier'):
                product_info['supplier'] = row_by_key['supplier']
            if row_by_key.get('product_number'):
                product_info['product_number'] = row_by_key['product_number']

        data = {
            'm_def': schema['m_def'],
            'name': sample_name,
            'datetime': date_picker.value.strftime('%Y-%m-%dT%H:%M:%S.%f'),
        }

        if dimensions_and_weights:
            data['dimensions_and_weights'] = dimensions_and_weights
        if product_info:
            data['product_info'] = product_info
        if row_by_key.get('description'):
            data['description'] = row_by_key['description']

        # Handle separator_stock reference
        if row_by_key.get('separator_stock'):
            sep_stock_input = row_by_key['separator_stock']
            # If it's already a reference (contains /archive/), use as-is
            # Otherwise, treat as a name and convert to reference
            if '/archive/' in str(sep_stock_input):
                data['separator_stock'] = sep_stock_input
            else:
                # Try to resolve the name to a reference
                sep_stock_ref = resolve_name_to_reference(sep_stock_input)
                if sep_stock_ref:
                    data['separator_stock'] = sep_stock_ref
                else:
                    row_errors.append(f"{sample_name}: Could not find SeparatorStock named '{sep_stock_input}'")

        file_name = f"{str(sample_name).replace(' ', '_')}.archive.json"
        prepared_rows.append({
            'row_number': row_number,
            'name': sample_name,
            'is_new': sample_name not in existing_ids,
            'file_name': file_name,
            'raw_path': file_name,
            'row_by_key': row_by_key,
            'archive': {'data': data},
        })

    return prepared_rows, row_errors

In [ ]:
def resolve_reference(reference_str):
    """
    Extract entry name from a NOMAD reference string.
    Input: ../uploads/{upload_id}/archive/{entry_id}#/data
    Returns: (entry_id, name) or (None, None) if not found
    """
    if not reference_str:
        return None, None
    
    try:
        # Parse reference: ../uploads/{upload_id}/archive/{entry_id}#/data
        parts = str(reference_str).split('/archive/')
        if len(parts) < 2:
            return None, None
        
        entry_id = parts[1].split('#')[0]
        if not entry_id:
            return None, None
        
        # Query by entry_id to get the name
        entries = iter_archive_query({'entry_id': entry_id})
        if not entries:
            return entry_id, None
        
        entry = entries[0]
        name = entry.get('archive', {}).get('data', {}).get('name')
        return entry_id, name
    
    except Exception as e:
        return None, None

def resolve_name_to_reference(sep_stock_name):
    """
    Find a SeparatorStock entry by name and return its reference string.
    Returns reference in format: ../uploads/{upload_id}/archive/{entry_id}#/data
    Strategy: Search current upload first, then all uploads
    """
    if not sep_stock_name:
        return None
    
    try:
        sep_stock_m_def = 'nomad_battery_space.schema_packages.hzb_bs_package.SeparatorStock'
        current_upload_id = get_upload_id(upload_dd.value) if upload_dd.value else None
        
        # Try current upload first
        if current_upload_id:
            current_entries = iter_archive_query({'upload_id': current_upload_id})
            matching = [
                e for e in current_entries
                if (e.get('archive', {}).get('data', {}).get('m_def') == sep_stock_m_def
                    and e.get('archive', {}).get('data', {}).get('name') == sep_stock_name)
            ]
            if matching:
                entry_id = matching[0].get('entry_id')
                reference = f"../uploads/{current_upload_id}/archive/{entry_id}#/data"
                #print(f"✓ Found SeparatorStock '{sep_stock_name}' in current upload: {reference}")
                return reference
        
        # Search all uploads if not found in current
        all_entries = iter_archive_query({})
        matching = [
            e for e in all_entries
            if (e.get('archive', {}).get('data', {}).get('m_def') == sep_stock_m_def
                and e.get('archive', {}).get('data', {}).get('name') == sep_stock_name)
        ]
        
        if not matching:
            print(f"⚠️ No SeparatorStock found with name: '{sep_stock_name}'")
            return None
        
        entry_id = matching[0].get('entry_id')
        entry_obj = matching[0]
        
        # Try to get upload_id from different possible locations
        upload_id = entry_obj.get('upload_id')  # Top-level field
        if not upload_id:
            metadata = entry_obj.get('metadata', {})
            upload_id = metadata.get('upload_id')  # In metadata
        if not upload_id:
            metadata = entry_obj.get('metadata', {})
            mainfile = metadata.get('mainfile', '')
            if mainfile:
                # mainfile format: uploads/{upload_id}/{path...}
                parts = mainfile.split('/')
                if parts[0] == 'uploads' and len(parts) > 1:
                    upload_id = parts[1]
        
        if not upload_id:
            print(f"⚠️ ERROR: Could not determine upload_id for '{sep_stock_name}' (entry_id: {entry_id})")
            return None
        
        reference = f"../uploads/{upload_id}/archive/{entry_id}#/data"
        #print(f"✓ Found SeparatorStock '{sep_stock_name}' in other upload: {reference}")
        return reference
    
    except Exception as e:
        print(f"⚠️ Error resolving name to reference for '{sep_stock_name}': {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
def render_status(kind, message):
    colors = {
        'error': '#f8d7da',
        'warning': '#fff3cd',
        'success': '#d4edda',
        'info': '#d1ecf1',
    }
    return HTML(
        f'<div style="background:{colors[kind]};padding:10px;border-radius:6px;margin-bottom:6px">{message}</div>'
    )

# Load uploads
uploads = get_uploads()
grid = None
col_keys = []
col_labels = []

upload_dd = widgets.Dropdown(
    options=[u.get('upload_name') for u in uploads if u.get('upload_name')],
    value=None,
    #description='NOMAD Upload:',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
)
schema_dd = widgets.Dropdown(
    options=list(SCHEMAS.keys()),
    value='Separator Sample',
    #description='Sample Type:',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
    disabled=True,
)
date_picker = widgets.NaiveDatetimePicker(
    #description='Sample Timestamp:',
    value=datetime.now(), 
    disabled=False,
    style={'description_width': 'initial', 'font_size': '16px'}
)

# Shape and Product Info selection
shape_dd = widgets.Dropdown(
    options=['Circle', 'Rectangle', 'Other'],
    value='Circle',
    #description='Shape:',
    layout=widgets.Layout(width='300px', height='40px'),
    style={'description_width': 'initial'},
)

product_info_cb = widgets.Checkbox(
    value=True,
    description='Include Product Information?',
    indent=False,
)

out_grid = widgets.Output()
out_existing = widgets.Output()
out_status = widgets.Output()

In [ ]:
def render_existing_samples(upload_id, schema):
    """Return HTML widget for displaying existing samples."""
    existing = get_existing_samples(upload_id, schema) if upload_id else []
    
    if not existing:
        return HTML('<p style="color: #999; font-style: italic;">No existing samples in this upload</p>')
    
    # Build existing entries table
    existing_data = []
    for sample in existing:
        data = sample.get('archive', {}).get('data', {})
        dims_weights = data.get('dimensions_and_weights', {})
        shape = dims_weights.get('shape', {}) if isinstance(dims_weights, dict) else {}
        product_info = data.get('product_info', {}) if isinstance(data.get('product_info'), dict) else {}
        
        # Resolve separator_stock reference to name
        sep_stock_ref = data.get('separator_stock')
        sep_stock_display = None
        if sep_stock_ref:
            _, sep_stock_name = resolve_reference(sep_stock_ref)
            sep_stock_display = sep_stock_name if sep_stock_name else sep_stock_ref
        
        row = {
            'Name': data.get('name', 'N/A'),
            'Separator Stock': sep_stock_display,
            'Thickness [µm]': dims_weights.get('thickness') if isinstance(dims_weights, dict) else None,
            'Mass [g]': dims_weights.get('mass') if isinstance(dims_weights, dict) else None,
            'Diameter [mm]': shape.get('diameter') if isinstance(shape, dict) else None,
            'Length [mm]': shape.get('length') if isinstance(shape, dict) else None,
            'Width [mm]': shape.get('width') if isinstance(shape, dict) else None,
            'Area [cm²]': shape.get('area') if isinstance(shape, dict) else None,
            'Supplier': product_info.get('supplier'),
            'Product Number': product_info.get('product_number'),
        }
        existing_data.append(row)
    
    df_existing = pd.DataFrame(existing_data)
    
    # Convert to HTML table (read-only)
    html_table = df_existing.fillna('').to_html(index=False, escape=False, border=0)
    html_table = f'''
    <div style="background: #f0f5ff; padding: 15px; border-radius: 8px; margin-bottom: 20px; border: 1px solid #ddd;">
        <h4 style="margin-top: 0; color: #333;">📋 Existing Samples in Upload</h4>
        <div style="overflow-x: auto; max-height: 300px; overflow-y: auto;">
            <table style="border-collapse: collapse; width: 100%; font-size: 0.9em;">
                {html_table.replace('<table>', '').replace('</table>', '')}
            </table>
        </div>
    </div>
    '''
    return HTML(html_table)

def show_grid():
    """Display existing samples (read-only) and new entry grid (editable)."""
    global grid, col_keys, col_labels
    
    schema_name = schema_dd.value
    schema = SCHEMAS[schema_name]
    upload_id = get_upload_id(upload_dd.value)
    
    # Get selected options
    selected_shape = shape_dd.value
    include_product_info = product_info_cb.value
    
    # Update schema columns dynamically
    schema['columns'] = get_columns_for_shape_and_product_info(selected_shape, include_product_info)
    
    col_keys = [column[0] for column in schema['columns']]
    col_labels = [column[1] for column in schema['columns']]
    
    # Display existing samples (read-only)
    out_existing.clear_output()
    with out_existing:
        widget_html = render_existing_samples(upload_id, schema)
        display(widget_html)
    
    # Create new entry grid
    df = pd.DataFrame(columns=col_labels)
    
    blank_rows = 12
    for _index in range(blank_rows):
        df.loc[len(df)] = pd.Series(dtype='object')
    
    grid_options = {
        'columnDefs': [{'headerName': label, 'field': label} for label in df.columns],
        'defaultColDef': {'editable': True, 'resizable': True},
        'rowSelection': 'multiple',
        'enableRangeSelection': True,
        'stopEditingWhenCellsLoseFocus': True,
    }
    grid = Grid(
        grid_data=df,
        grid_options=grid_options,
        sync_on_edit=True,
        theme='ag-theme-balham',
        columns_fit='auto',
        index=False,
    )
    
    out_grid.clear_output()
    with out_grid:
        display(grid)

In [ ]:
def on_create(_button):
    with out_status:
        clear_output(wait=True)
        
        schema_name = schema_dd.value
        schema = SCHEMAS[schema_name]
        upload_id = get_upload_id(upload_dd.value)
        
        if not upload_id:
            display(render_status('error', 'No upload selected'))
            return
        
        frame = extract_grid_frame()
        if frame.empty:
            display(render_status('warning', 'No data to upload'))
            return
        
        existing = get_existing_samples(upload_id, schema)
        existing_ids = set(s.get('archive', {}).get('data', {}).get('name') for s in existing if s.get('archive', {}).get('data', {}))
        
        prepared_rows, row_errors = prepare_rows_from_frame(frame, schema_name, schema, existing_ids)
        
        if row_errors:
            display(render_status('error', '<strong>Validation Errors:</strong><br>' + '<br>'.join(row_errors)))
            return
        
        if not prepared_rows:
            display(render_status('warning', 'No valid rows to upload'))
            return
        
        # Write archive bundle
        status_code, write_response = write_archive_bundle_via_api(upload_id, prepared_rows)
        
        write_errors = []
        if status_code != 200:
            error_msg = write_response if isinstance(write_response, str) else str(write_response)
            write_errors.append(f'Upload failed with status {status_code}: {error_msg}')
            display(render_status('error', '<strong>Upload failed:</strong><br>' + '<br>'.join(write_errors)))
            return
        
        created = len(prepared_rows)
        new_names = [r['name'] for r in prepared_rows if r.get('is_new')]
        
        display(render_status('info', f'Uploaded {created} archive files. Processing...'))
        
        # Process upload with retries
        missing_names = new_names.copy()
        for retry_idx in range(PROCESS_RETRY_COUNT):
            try:
                process_upload(upload_id, timeout=300)
                break
            except TimeoutError as e:
                if retry_idx < PROCESS_RETRY_COUNT - 1:
                    time.sleep(PROCESS_SETTLE_SECONDS)
                else:
                    display(render_status('warning', f'Processing timeout: {str(e)}'))
        
        # Verify new entries
        updated_existing = get_existing_samples(upload_id, schema)
        updated_existing_ids = set(s.get('archive', {}).get('data', {}).get('name') for s in updated_existing if s.get('archive', {}).get('data', {}))
        
        for name in new_names:
            if name in updated_existing_ids:
                missing_names.remove(name) if name in missing_names else None
        
        if created == 0 and not row_errors and not write_errors:
            display(render_status('warning', 'No rows were written. If you just edited the last cell, click outside the cell once and try again.'))
        elif created > 0:
            summary_parts = []
            if new_names:
                summary_parts.append(f'{len(new_names)} new')
            updated_count = created - len(new_names)
            if updated_count:
                summary_parts.append(f'{updated_count} updated')
            summary = ', '.join(summary_parts)
            display(render_status('success', f'✅ {summary} archive files uploaded into the NOMAD upload root.'))
            if missing_names:
                display(render_status('info', f"⚠️ Missing after processing: {', '.join(sorted(missing_names))}"))

In [ ]:
def on_context_change(_change):
    out_status.clear_output()
    show_grid()

upload_dd.observe(on_context_change, names='value')
schema_dd.observe(on_context_change, names='value')
shape_dd.observe(on_context_change, names='value')
product_info_cb.observe(on_context_change, names='value')

btn_create = widgets.Button(
    description='✅ Upload & Process',
    button_style='success',
    layout=widgets.Layout(width='180px', height='40px'),
    tooltip='Create sample entries from the data grid and trigger NOMAD processing',
)
btn_refresh = widgets.Button(
    description='🔄 Refresh',
    layout=widgets.Layout(width='110px', height='40px'),
    tooltip='Reload existing entries from NOMAD',
)

btn_create.on_click(on_create)
btn_refresh.on_click(lambda _: show_grid())

# Build the dashboard layout
init_message = widgets.HTML(
    '<div style="background: #e3f2fd; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; color: #0d47a1;">'
    '<strong>ℹ️ Please select an upload to begin</strong>'
    '</div>'
)

controls = widgets.VBox([
    widgets.HTML('<h3>Configuration</h3>'),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Type:</div>'),
        schema_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">NOMAD Upload:</div>'),
        upload_dd
    ]),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Timestamp:</div>'),
        date_picker
    ]),
])

new_samples_message = widgets.HTML(
    '<h3 style="background: #e3f2fd; color: #0d47a1; padding: 12px 16px; border-radius: 8px; border-left: 4px solid #1976d2; margin: 15px 0 8px; font-size: 1.3em;">📝 Register new Samples</h3>'
)

data_options = widgets.VBox([
    widgets.HTML('<h4 style="margin-bottom: 8px;">Data Options</h4>'),
    widgets.HBox([
        widgets.HTML('<div style="width: 180px; padding-top: 8px; font-weight: 500;">Sample Shape:</div>'),
        shape_dd
    ]),
    product_info_cb,
])

buttons = widgets.HBox([btn_create, btn_refresh])

dashboard = widgets.VBox([
    init_message,
    controls,
    out_existing,
    new_samples_message,
    data_options,
    out_grid,
    out_status,
    buttons,
])

# Display the complete dashboard
display(dashboard)

# Load grid initially with Circle shape
show_grid()

In [ ]:
def debug_prepared_rows(_button):
    with debug_output:
        clear_output(wait=True)
        schema_name = schema_dd.value
        schema = SCHEMAS[schema_name]
        upload_id = get_upload_id(upload_dd.value)
        
        if not upload_id:
            print("No upload selected")
            return
        
        # Get grid data 
        grid_data = extract_grid_frame()
        if grid_data.empty:
            print("Grid is empty")
            return
        
        existing_before = get_existing_samples(upload_id, schema)
        existing_ids = {
            normalize_sample_id(entry.get('archive', {}).get('data', {}).get('name'))
            for entry in existing_before
        }
        
        prepared_rows, row_errors = prepare_rows_from_frame(
            grid_data, schema_name, schema, existing_ids
        )
        
        if row_errors:
            print("⚠️ Row Errors:")
            for error in row_errors:
                print(f"  - {error}")
            print()
        
        print(f"Prepared {len(prepared_rows)} rows:\n")
        for i, row_data in enumerate(prepared_rows[:3]):  # First 3 rows
            archive_data = row_data.get('archive', {}).get('data', {})
            dims_weights = archive_data.get('dimensions_and_weights', {})
            shape = dims_weights.get('shape', {}) if isinstance(dims_weights, dict) else {}
            print(f"Row {i+1}: {row_data['name']}")
            print(f"  separator_stock: {archive_data.get('separator_stock')}")
            print(f"  dimensions_and_weights:")
            print(f"    - thickness: {dims_weights.get('thickness')}")
            print(f"    - mass: {dims_weights.get('mass')}")
            print(f"  shape (CircleGeometry):")
            print(f"    - diameter: {shape.get('diameter')}")
            print(f"    - area: (read-only, auto-calculated)")
            print()

btn_debug_prepared = widgets.Button(description='🔍 Prepared Rows JSON', layout=widgets.Layout(width='200px'))
btn_debug_prepared.on_click(debug_prepared_rows)

#display(widgets.HBox([btn_debug_prepared]))
#display(debug_output)